# Training dataset — build metadata CSV

Companion to `del/SVI_Val.ipynb`, which evaluates the **validation split** of the YOLO waste detector:

1. Read image paths from `dataSet/val.txt`
2. Map each image to its YOLO label in `labels/`
3. Compare ground-truth labels with model predictions (TP / FP / FN / TN)
4. Optionally run CPM on YOLO positives and export confusion matrices

This notebook starts the same way for the **training split**, but instead of model evaluation we build a tabular inventory like `Correct_SVI.csv` / the screenshot you shared:

| img_name | year | month | day | hour | lat | lon | panoid | img_dir | exist | collection | class | yolo_bbox | yolo_num |

Metadata comes from the existing `*_basic.csv` lookup tables (`Google_basic.csv`, `Faith_basic.csv`, `ZWL_basic.csv`). Bounding boxes are read from the training labels and converted to pixel coordinates (same list-of-lists format as `Correct_SVI.csv`). `yolo_conf` is omitted for now.

`collection` is the data source: **GSVI** if `img_name` ends with `_0`, `_90`, `_180`, or `_270` (Google Street View headings); otherwise **SC** (self-collected). This should align with `img_dir` (`Google/...` → GSVI; `Faith/` or `ZWL/` → SC).

`class` records each image's role in YOLO training: `waste` if the label file contains at least one bounding box, otherwise `background`.

In [2]:
import ast
import json
import os
import re
from pathlib import Path

import pandas as pd
from PIL import Image

In [3]:
DATA_ROOT = Path("/Users/wenlanzhang/Downloads/PhD_UCL/Data")
IMG_ROOT = DATA_ROOT / "Waste/img"
TRAIN_ROOT = IMG_ROOT / "Train0315_695"
OUTPUT_DIR = DATA_ROOT / "Chapter_waste" / "6_sensitivity"
OUTPUT_CSV = OUTPUT_DIR / "Train0315_695_training.csv"

SPLIT_FILE = TRAIN_ROOT / "dataSet/trainval.txt"  # all 696 labelled images; use train.txt for train-only
IMAGES_DIR = TRAIN_ROOT / "images"
LABELS_DIR = TRAIN_ROOT / "labels"

BASIC_CSVS = [
    IMG_ROOT / "Google_basic.csv",
    IMG_ROOT / "Faith_basic.csv",
    IMG_ROOT / "ZWL_basic.csv",
]

## 1. Load the training image list

`trainval.txt` lists every image used in the YOLO run (626 train + 70 val). Paths inside the file point at the older `GoogleStreetView/Train/Train0315/images/` tree, but the same basenames live locally under `Train0315_695/images/`.

In [4]:
with open(SPLIT_FILE) as f:
    split_paths = [line.strip() for line in f if line.strip()]

img_names = [Path(p).stem for p in split_paths]
print(f"Images in split file: {len(img_names)}")
print("Examples:", img_names[:5])

Images in split file: 696
Examples: ['LkN9dmH-f_QJFFxGOk7PEA_90', 'VSNF3NE5eykDgUUAqoayFg_270', '459eb9b0-UPoDfWE8S-Ya6uSXDf3w8Q_0', '701ab709-UrRoUBKATo0lZ4iSFwxUng_0', 'f10f8795-eIZ7kN3HHMhLCIFz3M5mNw_0']


## 2. Load metadata lookup tables

These are the same source tables used when building `Correct_SVI.csv` and `Combined_SVI.csv`.

In [5]:
meta_parts = []
for path in BASIC_CSVS:
    part = pd.read_csv(path)
    part["meta_source"] = path.stem.replace("_basic", "")
    meta_parts.append(part)

meta = pd.concat(meta_parts, ignore_index=True)
meta_by_name = meta.set_index("img_name", drop=False)
meta_by_panoid = meta.dropna(subset=["panoid"]).drop_duplicates("panoid", keep="first").set_index("panoid", drop=False)

print(meta["meta_source"].value_counts())
meta.head(3)

meta_source
Google    453364
ZWL         2930
Faith        817
Name: count, dtype: int64


,panoid,lat,lon,year,month,img_name,img_dir,exist,day,hour,meta_source
0,khlXWXsuIX7rodKwi1fDAw,-1.347984,36.904885,2021,10,khlXWXsuIX7rodKwi1fDAw_0,Google/k/h/,True,NaN,NaN,Google
1,khlXWXsuIX7rodKwi1fDAw,-1.347984,36.904885,2021,10,khlXWXsuIX7rodKwi1fDAw_90,Google/k/h/,True,NaN,NaN,Google
2,khlXWXsuIX7rodKwi1fDAw,-1.347984,36.904885,2021,10,khlXWXsuIX7rodKwi1fDAw_180,Google/k/h/,True,NaN,NaN,Google


## 3. Helper functions

In [6]:
UUID_PREFIX = re.compile(r"^[0-9a-f]{8}-", re.I)
GSVI_SUFFIX = re.compile(r"_(0|90|180|270)$")
JUNK_NAMES = {".DS_Store"}


def collection_from_name(img_name: str) -> str:
    """GSVI if the filename ends with a Street View heading; otherwise self-collected."""
    return "GSVI" if GSVI_SUFFIX.search(img_name) else "SC"


def collection_from_img_dir(img_dir) -> str | None:
    if pd.isna(img_dir):
        return None
    path = str(img_dir)
    if path.startswith("Google/"):
        return "GSVI"
    if path.startswith("Faith/") or path.startswith("ZWL/"):
        return "SC"
    return None


def candidate_names(img_name: str) -> list[str]:
    """Try direct name, then strip an 8-char hex prefix used on some self-collected files."""
    names = [img_name]
    if UUID_PREFIX.match(img_name):
        names.append(img_name.split("-", 1)[1])
    return list(dict.fromkeys(names))


def panoid_from_name(img_name: str) -> str:
    base = img_name.split("-", 1)[1] if UUID_PREFIX.match(img_name) else img_name
    if "_" in base:
        return base.rsplit("_", 1)[0]
    return base


def lookup_metadata(img_name: str) -> dict | None:
    for cand in candidate_names(img_name):
        if cand in meta_by_name.index:
            row = meta_by_name.loc[cand]
            return row.iloc[0].to_dict() if isinstance(row, pd.DataFrame) else row.to_dict()

    panoid = panoid_from_name(img_name)
    if panoid in meta_by_panoid.index:
        return meta_by_panoid.loc[panoid].to_dict()
    return None


def yolo_to_pixel_bbox(line: str, width: int, height: int) -> list[float]:
    """Convert one YOLO label line (class cx cy w h, normalised) to [x1, y1, x2, y2] pixels."""
    parts = line.strip().split()
    _, cx, cy, bw, bh = map(float, parts[:5])
    x1 = (cx - bw / 2) * width
    y1 = (cy - bh / 2) * height
    x2 = (cx + bw / 2) * width
    y2 = (cy + bh / 2) * height
    return [x1, y1, x2, y2]


def read_label_bboxes(img_name: str) -> tuple[list[list[float]], int]:
    label_path = LABELS_DIR / f"{img_name}.txt"
    image_path = IMAGES_DIR / f"{img_name}.jpg"

    if not label_path.exists() or label_path.stat().st_size == 0:
        return [], 0

    if not image_path.exists():
        return [], 0

    with Image.open(image_path) as im:
        width, height = im.size

    boxes = []
    with open(label_path) as f:
        for line in f:
            line = line.strip()
            if line:
                boxes.append(yolo_to_pixel_bbox(line, width, height))
    return boxes, len(boxes)

## 4. Build the training table

In [7]:
rows = []
missing_meta = []
missing_image = []

for img_name in img_names:
    if img_name in JUNK_NAMES:
        continue

    image_path = IMAGES_DIR / f"{img_name}.jpg"
    if not image_path.exists():
        missing_image.append(img_name)

    meta_row = lookup_metadata(img_name)
    if meta_row is None:
        missing_meta.append(img_name)
        meta_row = {
            "year": pd.NA,
            "month": pd.NA,
            # "day": pd.NA,
            # "hour": pd.NA,
            "lat": pd.NA,
            "lon": pd.NA,
            "panoid": pd.NA,
            "img_dir": pd.NA,
            "exist": False,
        }

    yolo_bbox, yolo_num = read_label_bboxes(img_name)
    training_class = "waste" if yolo_num > 0 else "background"
    collection = collection_from_name(img_name)

    rows.append(
        {
            "img_name": img_name,
            "year": meta_row.get("year"),
            "month": meta_row.get("month"),
            # "day": meta_row.get("day"),
            # "hour": meta_row.get("hour"),
            "lat": meta_row.get("lat"),
            "lon": meta_row.get("lon"),
            "panoid": meta_row.get("panoid"),
            "img_dir": meta_row.get("img_dir"),
            "exist": bool(image_path.exists()) if image_path.exists() else meta_row.get("exist", False),
            "collection": collection,
            "class": training_class,
            # "yolo_bbox": json.dumps(yolo_bbox),
            # "yolo_num": yolo_num,
        }
    )

train_df = pd.DataFrame(rows)
print(f"Rows built: {len(train_df)}")
print("\nCollection:")
print(train_df["collection"].value_counts().to_string())
print("\nClass:")
print(train_df["class"].value_counts().to_string())
print(f"Missing metadata: {len(missing_meta)}")
print(f"Missing local image file: {len(missing_image)}")
if missing_meta:
    print("Unmatched img_name examples:", missing_meta[:10])
train_df.head()

Rows built: 695

Collection:
collection
GSVI    610
SC       85

Class:
class
waste         353
background    342
Missing metadata: 8
Missing local image file: 0
Unmatched img_name examples: ['1f7e5b1e-3767618616712176', '12fe849c-WechatIMG1048', 'b177368d-WechatIMG1047', 'e46a1e51-WechatIMG1046', '44c2da02-IMG_6653', 'aa90b6e2-WechatIMG1049', 'dddd727b-IMG_6652', '37eb2376-310211603998748']


,img_name,year,month,lat,lon,panoid,img_dir,exist,collection,class
0,LkN9dmH-f_QJFFxGOk7PEA_90,2018,10,-1.374241,36.779327,LkN9dmH-f_QJFFxGOk7PEA,Google/L/k/,True,GSVI,background
1,VSNF3NE5eykDgUUAqoayFg_270,2021,7,-1.238584,36.851001,VSNF3NE5eykDgUUAqoayFg,Google/V/S/,True,GSVI,background
2,459eb9b0-UPoDfWE8S-Ya6uSXDf3w8Q_0,2018,4,-1.320441,36.892761,UPoDfWE8S-Ya6uSXDf3w8Q,Google/U/P/,True,GSVI,waste
3,701ab709-UrRoUBKATo0lZ4iSFwxUng_0,2021,8,-1.312264,36.857056,UrRoUBKATo0lZ4iSFwxUng,Google/U/r/,True,GSVI,waste
4,f10f8795-eIZ7kN3HHMhLCIFz3M5mNw_0,2021,9,-1.186151,36.905804,eIZ7kN3HHMhLCIFz3M5mNw,Google/e/I/,True,GSVI,waste


## 5. Quick checks

In [8]:
from IPython.display import Markdown, display

n = len(train_df)

# Marginal counts
collection_tbl = (
    train_df["collection"]
    .value_counts()
    .reindex(["GSVI", "SC"])
    .rename("Count")
    .to_frame()
    .assign(**{"% of total": lambda d: (d["Count"] / n * 100).round(1)})
)
collection_tbl["Count"] = collection_tbl["Count"].map("{:,}".format)

class_tbl = (
    train_df["class"]
    .value_counts()
    .reindex(["waste", "background"])
    .rename("Count")
    .to_frame()
    .assign(**{"% of total": lambda d: (d["Count"] / n * 100).round(1)})
)
class_tbl["Count"] = class_tbl["Count"].map("{:,}".format)

# Stratified table: counts + within-collection %
counts = pd.crosstab(train_df["collection"], train_df["class"]).reindex(index=["GSVI", "SC"], columns=["background", "waste"])
within_pct = counts.div(counts.sum(axis=1), axis=0).mul(100).round(1)

stratified_tbl = counts.copy()
stratified_tbl["Total"] = counts.sum(axis=1)
stratified_tbl["Waste %"] = within_pct["waste"]
stratified_tbl.loc["Total"] = [
    counts["background"].sum(),
    counts["waste"].sum(),
    n,
    (train_df["class"] == "waste").mean() * 100,
]

display(Markdown(f"### Training set summary — **{n:,}** images"))
display(Markdown("**By collection**"))
display(collection_tbl)
display(Markdown("**By YOLO class**"))
display(class_tbl)
display(Markdown("**Stratified: collection × class**"))
display(
    stratified_tbl.rename_axis("Collection")
    .style.format({
        "background": "{:,.0f}",
        "waste": "{:,.0f}",
        "Total": "{:,.0f}",
        "Waste %": "{:.1f}%",
    }, na_rep="")
    .set_caption("Counts with waste share within each collection")
)

### Training set summary — **695** images

**By collection**

,Count,% of total
collection,,
GSVI,610,87.8
SC,85,12.2


**By YOLO class**

,Count,% of total
class,,
waste,353,50.8
background,342,49.2


**Stratified: collection × class**

class,background,waste,Total,Waste %
Collection,,,,
GSVI,295,315,610,51.6%
SC,47,38,85,44.7%
Total,342,353,695,50.8%


In [9]:
from IPython.display import Markdown, display

train_df["collection_from_dir"] = train_df["img_dir"].map(collection_from_img_dir)
known = train_df["collection_from_dir"].notna()
mismatches = train_df.loc[known & (train_df["collection"] != train_df["collection_from_dir"])]
missing_dir = train_df[train_df["collection_from_dir"].isna()]

# QA headline
qa_tbl = pd.DataFrame(
    {
        "Check": [
            "Rows with known img_dir",
            "Name rule matches img_dir",
            "Mismatches",
            "Missing img_dir",
        ],
        "Result": [
            f"{known.sum():,} / {len(train_df):,}",
            f"{known.sum() - len(mismatches):,} / {known.sum():,}",
            f"{len(mismatches):,}",
            f"{len(missing_dir):,}",
        ],
        "Status": [
            "OK" if known.sum() > 0 else "—",
            "OK" if len(mismatches) == 0 else "Review",
            "OK" if len(mismatches) == 0 else "Review",
            "Note" if len(missing_dir) > 0 else "OK",
        ],
    }
)

# collection (name) vs collection (img_dir)
check_df = train_df.assign(
    collection_from_dir=train_df["collection_from_dir"].fillna("missing img_dir")
)
cross_tbl = (
    pd.crosstab(check_df["collection"], check_df["collection_from_dir"])
    .reindex(index=["GSVI", "SC"], fill_value=0)
)

# Top img_dir roots
img_dir_tbl = (
    train_df["img_dir"]
    .fillna("<missing>")
    .map(lambda x: "Google/…" if str(x).startswith("Google/") else x)
    .value_counts()
    .head(10)
    .rename("Count")
    .to_frame()
    .assign(**{"% of total": lambda d: (d["Count"] / len(train_df) * 100).round(1)})
)

display(Markdown("### Collection consistency check"))
display(Markdown("_Does the filename rule (`_0/_90/_180/_270` → GSVI) agree with `img_dir`?_"))
display(qa_tbl.style.hide(axis="index"))

display(Markdown("**Name rule vs `img_dir`**"))
display(
    cross_tbl.rename_axis("From img_name")
    .rename_axis("From img_dir", axis=1)
    .style.format("{:,.0f}")
    .set_caption("Cross-tab: collection from filename vs collection inferred from img_dir")
)

if len(missing_dir):
    display(Markdown(f"**Missing `img_dir`** ({len(missing_dir)} rows)"))
    display(
        missing_dir[["img_name", "collection"]]
        .reset_index(drop=True)
        .style.hide(axis="index")
    )

display(Markdown("**Top `img_dir` groups**"))
display(img_dir_tbl.style.format({"Count": "{:,.0f}", "% of total": "{:.1f}%"}))

### Collection consistency check

_Does the filename rule (`_0/_90/_180/_270` → GSVI) agree with `img_dir`?_

Check,Result,Status
Rows with known img_dir,687 / 695,OK
Name rule matches img_dir,687 / 687,OK
Mismatches,0,OK
Missing img_dir,8,Note


**Name rule vs `img_dir`**

From img_dir,GSVI,SC,missing img_dir
From img_name,,,
GSVI,610,0,0
SC,0,77,8


**Missing `img_dir`** (8 rows)

img_name,collection
1f7e5b1e-3767618616712176,SC
12fe849c-WechatIMG1048,SC
b177368d-WechatIMG1047,SC
e46a1e51-WechatIMG1046,SC
44c2da02-IMG_6653,SC
aa90b6e2-WechatIMG1049,SC
dddd727b-IMG_6652,SC
37eb2376-310211603998748,SC


**Top `img_dir` groups**

,Count,% of total
img_dir,,
Google/…,610,87.8%
ZWL/,77,11.1%
,8,1.2%


## 6. Export CSV

In [10]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
export_df = train_df.drop(columns=["collection_from_dir"], errors="ignore")
export_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(export_df):,} rows to {OUTPUT_CSV}")

Saved 695 rows to /Users/wenlanzhang/PycharmProjects/Chapter_waste/6_sensitivity/Train0315_695_training.csv


## 7. GSVI ↔ SC swap scenarios (always 695 images)

Current training = **610 GSVI (fixed)** + **85 SC (switchable)**. Every scenario keeps **695 images** — we only swap some of the **85 SC images** for **GSVI replacements** (waste↔waste, background↔background).

| `sc_share_pct` | Meaning | Example |
|---|---|---|
| **100%** | All 85 SC kept → **current baseline** | 85 SC / 0 swapped |
| **50%** | Half of the 85 SC slots swapped to GSVI | ~43 SC / ~42 swapped |
| **0%** | All 85 SC swapped → **695 GSVI** | 0 SC / 85 swapped |

The 610 GSVI core never changes. Scenarios: **0%, 25%, 50%, 75%, 100%**.

Exports per scenario (`sc_0pct`, `sc_50pct`, `sc_100pct`, …):

- `*_training.csv` — 695-row image list
- `*_train.txt` — YOLO paths (local images only)


In [11]:
CORRECT_SVI_CSV = IMG_ROOT / "Correct_SVI.csv"
SENSITIVITY_CSV = OUTPUT_DIR / "Train0315_695_sensitivity_inventory.csv"
SCENARIO_DIR = OUTPUT_DIR / "scenarios"
MANIFEST_CSV = OUTPUT_DIR / "Train0315_695_gsvi_sensitivity_manifest.csv"
POOLS_SUMMARY_CSV = OUTPUT_DIR / "Train0315_695_pools_summary.csv"

SC_SHARE_PCTS = [0, 25, 50, 75, 100]
RANDOM_SEED = 42


def build_name_lookup(names: pd.Series) -> dict[str, str]:
    lookup: dict[str, str] = {}
    for canonical in names:
        for cand in candidate_names(canonical):
            lookup.setdefault(cand, canonical)
    return lookup


def match_lookup_name(img_name: str, lookup: dict[str, str]) -> str | None:
    for cand in candidate_names(img_name):
        if cand in lookup:
            return lookup[cand]
    return None


def sensitivity_role(in_training: bool, training_class: str, svi_waste: bool) -> str:
    if in_training and training_class == "waste" and svi_waste:
        return "training_waste_confirmed_svi"
    if in_training and training_class == "waste":
        return "training_waste_only"
    if in_training:
        return "training_background"
    return "svi_unused_waste"


COMBINED_SVI_CSV = IMG_ROOT / "Combined_SVI.csv"
TOTAL_IMAGES = 695
WASTE_TOTAL = 353
BG_TOTAL = 342


def has_local_files(img_name: str) -> bool:
    return (IMAGES_DIR / f"{img_name}.jpg").exists() and (LABELS_DIR / f"{img_name}.txt").exists()


def sc_slot_counts(sc_share_pct: int) -> dict[str, int]:
    """Of the 85 SC slots, how many stay SC vs swap to GSVI (split by class)."""
    sc_waste_keep = round(SC_WASTE_SLOTS * sc_share_pct / 100)
    sc_bg_keep = round(SC_BG_SLOTS * sc_share_pct / 100)
    return {
        "sc_waste_keep": sc_waste_keep,
        "sc_waste_swap": SC_WASTE_SLOTS - sc_waste_keep,
        "sc_bg_keep": sc_bg_keep,
        "sc_bg_swap": SC_BG_SLOTS - sc_bg_keep,
        "sc_total_keep": sc_waste_keep + sc_bg_keep,
        "sc_total_swap": (SC_WASTE_SLOTS - sc_waste_keep) + (SC_BG_SLOTS - sc_bg_keep),
    }


def pick_training_rows(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if n == 0:
        return df.iloc[0:0].copy()
    if n >= len(df):
        return df.copy()
    return df.sample(n=n, random_state=seed).copy()


def pick_from_pool(pool: pd.DataFrame, n: int, seed: int, pool_name: str) -> pd.DataFrame:
    if n > len(pool):
        raise ValueError(f"Need {n:,} images from {pool_name}, but only {len(pool):,} available")

    local = pool[pool["has_local"]]
    non_local = pool[~pool["has_local"]]

    if n <= len(local):
        return local.sample(n=n, random_state=seed).copy()

    picked = [local]
    remaining = n - len(local)
    if remaining > len(non_local):
        raise ValueError(
            f"Need {remaining:,} extra non-local images from {pool_name}, but only {len(non_local):,} available"
        )
    picked.append(non_local.sample(n=remaining, random_state=seed))
    return pd.concat(picked, ignore_index=True)


def build_candidate_pool(collection: str, cls: str, training_df: pd.DataFrame, extras_df: pd.DataFrame) -> pd.DataFrame:
    train_part = training_df[(training_df["collection"] == collection) & (training_df["class"] == cls)].copy()
    train_part["candidate_source"] = "training"

    extra_part = extras_df[(extras_df["collection"] == collection) & (extras_df["class"] == cls)].copy()
    extra_part["candidate_source"] = "extra"

    pool = pd.concat([train_part, extra_part], ignore_index=True)
    pool = pool.drop_duplicates("img_name", keep="first")
    pool["has_local"] = pool["img_name"].map(has_local_files)
    pool = pool.sort_values(["has_local", "candidate_source"], ascending=[False, True])
    return pool


# --- Join training to SVI-identified waste inventory ---
svi_waste = pd.read_csv(CORRECT_SVI_CSV)
svi_waste["collection"] = svi_waste["img_name"].map(collection_from_name)
dir_collection = svi_waste["img_dir"].map(collection_from_img_dir)
svi_waste.loc[svi_waste["collection"].isna(), "collection"] = dir_collection

svi_lookup = build_name_lookup(svi_waste["img_name"])

train_cmp = train_df.drop(columns=["collection_from_dir"], errors="ignore").copy()
train_cmp["svi_img_name"] = train_cmp["img_name"].map(lambda n: match_lookup_name(n, svi_lookup))
train_cmp["svi_waste"] = train_cmp["svi_img_name"].notna()
train_cmp["in_training"] = True
train_cmp["sensitivity_role"] = train_cmp.apply(
    lambda r: sensitivity_role(True, r["class"], r["svi_waste"]),
    axis=1,
)

matched_svi_names = set(train_cmp["svi_img_name"].dropna())
svi_unused = svi_waste[~svi_waste["img_name"].isin(matched_svi_names)].copy()
svi_unused["class"] = "waste"
svi_unused["in_training"] = False
svi_unused["svi_waste"] = True
svi_unused["svi_img_name"] = svi_unused["img_name"]
svi_unused["sensitivity_role"] = "svi_unused_waste"

SHARED_COLS = [
    "img_name", "year", "month", "lat", "lon", "panoid", "img_dir", "exist",
    "collection", "class", "in_training", "svi_waste", "svi_img_name", "sensitivity_role",
]

sensitivity_df = pd.concat(
    [train_cmp[SHARED_COLS], svi_unused[SHARED_COLS]],
    ignore_index=True,
)

# --- Swap candidates: verified waste from Correct_SVI; background from Combined_SVI (non-waste) ---
correct_waste_names = set(svi_waste["img_name"])

waste_extra = svi_unused.copy()
waste_extra["svi_img_name"] = waste_extra["img_name"]
waste_extra["sensitivity_role"] = "correct_svi_waste"

combined = pd.read_csv(COMBINED_SVI_CSV, low_memory=False)
combined["collection"] = combined["img_name"].map(collection_from_name)
combined.loc[combined["collection"].isna(), "collection"] = combined["img_dir"].map(collection_from_img_dir)

background_extra = combined[
    ~combined["img_name"].isin(correct_waste_names)
    & ~combined["img_name"].isin(train_cmp["img_name"])
].copy()
background_extra["class"] = "background"
background_extra["svi_waste"] = False
background_extra["svi_img_name"] = pd.NA
background_extra["in_training"] = False
background_extra["sensitivity_role"] = "combined_svi_background"

extras = pd.concat(
    [waste_extra[SHARED_COLS], background_extra[SHARED_COLS]],
    ignore_index=True,
).drop_duplicates("img_name", keep="first")

POOL_DEFS = [
    ("gsvi_waste", "GSVI", "waste"),
    ("gsvi_background", "GSVI", "background"),
    ("sc_waste", "SC", "waste"),
    ("sc_background", "SC", "background"),
]
POOL_SOURCES = {
    "gsvi_waste": "Correct_SVI.csv",
    "gsvi_background": "Combined_SVI.csv (not in Correct_SVI)",
    "sc_waste": "Correct_SVI.csv",
    "sc_background": "Combined_SVI.csv (not in Correct_SVI)",
}

candidate_pools: dict[str, pd.DataFrame] = {}
for pool_name, collection, cls in POOL_DEFS:
    pool = build_candidate_pool(collection, cls, train_cmp, extras)
    pool["pool"] = pool_name
    candidate_pools[pool_name] = pool

pools_summary = pd.DataFrame(
    {
        "pool": [name for name, _, _ in POOL_DEFS],
        "source": [POOL_SOURCES[name] for name, _, _ in POOL_DEFS],
        "collection": [c for _, c, _ in POOL_DEFS],
        "class": [cls for _, _, cls in POOL_DEFS],
        "in_training": [
            (candidate_pools[name]["candidate_source"] == "training").sum() for name, _, _ in POOL_DEFS
        ],
        "extra_candidates": [
            (candidate_pools[name]["candidate_source"] == "extra").sum() for name, _, _ in POOL_DEFS
        ],
        "total": [len(candidate_pools[name]) for name, _, _ in POOL_DEFS],
        "local": [candidate_pools[name]["has_local"].sum() for name, _, _ in POOL_DEFS],
    }
)

# --- Fixed GSVI core + switchable SC slots ---
gsvi_core = train_cmp[train_cmp["collection"] == "GSVI"].copy()
sc_waste_training = train_cmp[(train_cmp["collection"] == "SC") & (train_cmp["class"] == "waste")].copy()
sc_bg_training = train_cmp[(train_cmp["collection"] == "SC") & (train_cmp["class"] == "background")].copy()

SC_WASTE_SLOTS = len(sc_waste_training)
SC_BG_SLOTS = len(sc_bg_training)
SC_SLOTS = SC_WASTE_SLOTS + SC_BG_SLOTS
GSVI_CORE_SLOTS = len(gsvi_core)

core_names = set(gsvi_core["img_name"])
replacement_pools = {
    "gsvi_waste": waste_extra[
        (waste_extra["collection"] == "GSVI") & (~waste_extra["img_name"].isin(core_names))
    ].copy(),
    "gsvi_background": background_extra[
        (background_extra["collection"] == "GSVI") & (~background_extra["img_name"].isin(core_names))
    ].copy(),
}
for pool_name, pool_df in replacement_pools.items():
    pool_df["has_local"] = pool_df["img_name"].map(has_local_files)

print(f"Fixed GSVI core: {GSVI_CORE_SLOTS:,} images")
print(f"Switchable SC slots: {SC_SLOTS:,} ({SC_WASTE_SLOTS:,} waste + {SC_BG_SLOTS:,} background)")
print(f"Target per scenario: {TOTAL_IMAGES:,} images ({WASTE_TOTAL:,} waste + {BG_TOTAL:,} background)")
print("\nVerified waste = Correct_SVI.csv only (not yolo_num)")
print(f"  Correct_SVI total: {len(svi_waste):,} | not in training: {len(waste_extra):,}")
print("\nGSVI replacement pool (for swapped SC slots):")
print(
    f"  waste (Correct_SVI): {len(replacement_pools['gsvi_waste']):,} candidates "
    f"({replacement_pools['gsvi_waste']['has_local'].sum():,} local)"
)
print(
    f"  background (Combined_SVI, not in Correct_SVI): {len(replacement_pools['gsvi_background']):,} candidates "
    f"({replacement_pools['gsvi_background']['has_local'].sum():,} local)"
)
print("\nSwap candidate pools (training + extras):")
print(pools_summary.to_string(index=False))

Fixed GSVI core: 610 images
Switchable SC slots: 85 (38 waste + 47 background)
Target per scenario: 695 images (353 waste + 342 background)

Verified waste = Correct_SVI.csv only (not yolo_num)
  Correct_SVI total: 4,082 | not in training: 3,749

GSVI replacement pool (for swapped SC slots):
  waste (Correct_SVI): 3,595 candidates (0 local)
  background (Combined_SVI, not in Correct_SVI): 449,173 candidates (0 local)

Swap candidate pools (training + extras):
           pool                                source collection      class  in_training  extra_candidates  total  local
     gsvi_waste                       Correct_SVI.csv       GSVI      waste          315              3595   3910    315
gsvi_background Combined_SVI.csv (not in Correct_SVI)       GSVI background          295            449173 449468    295
       sc_waste                       Correct_SVI.csv         SC      waste           38               154    192     38
  sc_background Combined_SVI.csv (not in Correct_SVI

In [12]:
# from IPython.display import Markdown, display

# IMAGE_PATH_PREFIX = TRAIN_ROOT / "images"
# baseline_names = set(train_cmp["img_name"])


# def build_swap_scenario(sc_share_pct: int, seed: int = RANDOM_SEED) -> pd.DataFrame:
#     """Keep all 610 GSVI; swap a fraction of the 85 SC slots to GSVI replacements."""
#     counts = sc_slot_counts(sc_share_pct)
#     parts = []

#     core = gsvi_core.copy()
#     core["pool"] = core["class"].map({"waste": "gsvi_waste", "background": "gsvi_background"})
#     core["swap_role"] = "gsvi_core"
#     core["candidate_source"] = "training"
#     parts.append(core)

#     sc_waste_kept = pick_training_rows(sc_waste_training, counts["sc_waste_keep"], seed)
#     sc_waste_kept["pool"] = "sc_waste"
#     sc_waste_kept["swap_role"] = "sc_kept"
#     sc_waste_kept["candidate_source"] = "training"
#     parts.append(sc_waste_kept)

#     sc_bg_kept = pick_training_rows(sc_bg_training, counts["sc_bg_keep"], seed)
#     sc_bg_kept["pool"] = "sc_background"
#     sc_bg_kept["swap_role"] = "sc_kept"
#     sc_bg_kept["candidate_source"] = "training"
#     parts.append(sc_bg_kept)

#     if counts["sc_waste_swap"]:
#         repl_w = pick_from_pool(replacement_pools["gsvi_waste"], counts["sc_waste_swap"], seed, "gsvi_waste_replace")
#         repl_w["pool"] = "gsvi_waste"
#         repl_w["swap_role"] = "gsvi_replacement"
#         parts.append(repl_w)

#     if counts["sc_bg_swap"]:
#         repl_b = pick_from_pool(replacement_pools["gsvi_background"], counts["sc_bg_swap"], seed, "gsvi_background_replace")
#         repl_b["pool"] = "gsvi_background"
#         repl_b["swap_role"] = "gsvi_replacement"
#         parts.append(repl_b)

#     scenario = pd.concat(parts, ignore_index=True)
#     scenario["sc_share_pct"] = sc_share_pct
#     scenario["scenario"] = f"sc_{sc_share_pct}pct"
#     scenario["has_local"] = scenario["img_name"].map(has_local_files)

#     assert len(scenario) == TOTAL_IMAGES
#     assert (scenario["class"] == "waste").sum() == WASTE_TOTAL
#     assert (scenario["class"] == "background").sum() == BG_TOTAL
#     return scenario


# def swaps_vs_baseline(scenario_df: pd.DataFrame) -> tuple[int, int]:
#     scenario_names = set(scenario_df["img_name"])
#     out_only = baseline_names - scenario_names
#     in_only = scenario_names - baseline_names
#     return len(out_only), len(in_only)


# scenario_tables: dict[str, pd.DataFrame] = {}
# scenario_rows = []

# for pct in SC_SHARE_PCTS:
#     scenario_df = build_swap_scenario(pct)
#     scenario_tables[f"sc_{pct}pct"] = scenario_df
#     swapped_out, swapped_in = swaps_vs_baseline(scenario_df)
#     counts = sc_slot_counts(pct)

#     scenario_rows.append(
#         {
#             "scenario": f"sc_{pct}pct",
#             "sc_share_pct": pct,
#             "sc_kept": counts["sc_total_keep"],
#             "sc_swapped_to_gsvi": counts["sc_total_swap"],
#             "gsvi_images": (scenario_df["collection"] == "GSVI").sum(),
#             "sc_images": (scenario_df["collection"] == "SC").sum(),
#             "gsvi_core": GSVI_CORE_SLOTS,
#             "gsvi_replacements": (scenario_df["swap_role"] == "gsvi_replacement").sum(),
#             "total_images": len(scenario_df),
#             "swapped_out": swapped_out,
#             "swapped_in": swapped_in,
#             "local_images": scenario_df["has_local"].sum(),
#         }
#     )

# scenario_summary = pd.DataFrame(scenario_rows)

# display(Markdown("### SC-slot swap scenarios (always 695 images)"))
# display(Markdown("_610 GSVI fixed. Only the 85 SC positions change: 100% = baseline, 0% = all 695 GSVI._"))
# display(
#     scenario_summary.style.format(
#         {
#             "sc_share_pct": "{:.0f}",
#             "sc_kept": "{:,.0f}",
#             "sc_swapped_to_gsvi": "{:,.0f}",
#             "gsvi_images": "{:,.0f}",
#             "sc_images": "{:,.0f}",
#             "gsvi_core": "{:,.0f}",
#             "gsvi_replacements": "{:,.0f}",
#             "total_images": "{:,.0f}",
#             "swapped_out": "{:,.0f}",
#             "swapped_in": "{:,.0f}",
#             "local_images": "{:,.0f}",
#         }
#     ).hide(axis="index")
# )

# pct100_names = set(scenario_tables["sc_100pct"]["img_name"])
# print(f"sc_100pct matches current training: {baseline_names == pct100_names}")
# print(f"sc_50pct example: {scenario_summary.loc[scenario_summary['sc_share_pct']==50, 'sc_kept'].iloc[0]:.0f} SC kept / {scenario_summary.loc[scenario_summary['sc_share_pct']==50, 'sc_swapped_to_gsvi'].iloc[0]:.0f} swapped")

In [13]:
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# SCENARIO_DIR.mkdir(parents=True, exist_ok=True)

# manifest_parts = []
# export_cols = [
#     "scenario", "sc_share_pct", "swap_role", "pool", "collection", "class", "candidate_source", "has_local",
#     "img_name", "year", "month", "lat", "lon", "panoid", "img_dir", "exist",
#     "svi_waste", "sensitivity_role",
# ]

# for tag, scenario_df in scenario_tables.items():
#     csv_path = SCENARIO_DIR / f"{tag}_training.csv"
#     txt_path = SCENARIO_DIR / f"{tag}_train.txt"

#     scenario_df[export_cols].to_csv(csv_path, index=False)

#     local_df = scenario_df[scenario_df["has_local"]]
#     image_paths = [str(IMAGE_PATH_PREFIX / f"{name}.jpg") for name in local_df["img_name"]]
#     txt_path.write_text("\n".join(image_paths) + "\n")

#     manifest_parts.append(scenario_df[export_cols])
#     print(
#         f"{tag}: {len(scenario_df):,} images | local in train.txt: {len(local_df):,} | "
#         f"GSVI {(scenario_df['collection']=='GSVI').sum():,} / SC {(scenario_df['collection']=='SC').sum():,}"
#     )

# manifest_df = pd.concat(manifest_parts, ignore_index=True)

# sensitivity_df.to_csv(SENSITIVITY_CSV, index=False)
# pools_summary.to_csv(POOLS_SUMMARY_CSV, index=False)
# scenario_summary.to_csv(OUTPUT_DIR / "Train0315_695_gsvi_scenario_summary.csv", index=False)
# manifest_df.to_csv(MANIFEST_CSV, index=False)

# print(f"\nSaved files under {SCENARIO_DIR}/ and summary CSVs in {OUTPUT_DIR}/")

## 8. GSVI images to label for sensitivity swaps

The 610 GSVI core is already labelled. For swapped SC slots we need **extra GSVI replacements** — counted separately for **waste** and **background**.

**Label-first workflow:** scenarios are built *after* labelling. This cell only prepares what to label now.

1. Shows per-scenario swap counts (planning reference — worst case = `sc_0pct`)
2. Picks the **required** max replacements directly from `replacement_pools` (38 waste + 47 background)
3. Adds a **buffer** (default 20%) for misclassification
4. Exports a copy manifest for `sensitivity/waste` and `sensitivity/background`

Run section 7 (pool summary) first so `replacement_pools` exists.

In [16]:
from IPython.display import Markdown, display
import math

SENSITIVITY_IMG_ROOT = IMG_ROOT / "sensitivity"
SENSITIVITY_WASTE_DIR = SENSITIVITY_IMG_ROOT / "waste"
SENSITIVITY_BG_DIR = SENSITIVITY_IMG_ROOT / "background"
LABEL_MANIFEST_CSV = OUTPUT_DIR / "Train0315_695_gsvi_label_manifest.csv"
SWAP_NEEDS_CSV = OUTPUT_DIR / "Train0315_695_gsvi_swap_needs.csv"

LABEL_BUFFER_PCT = 0.40  # 20% extra; set to 0.10 for 10% buffer


def buffered_count(n: int, buffer_pct: float) -> int:
    return math.ceil(n * (1 + buffer_pct))


def gsvi_archive_path(img_name: str, img_dir: str) -> Path:
    """Google Street View image on disk: IMG_ROOT / img_dir / img_name.jpg"""
    return (IMG_ROOT / str(img_dir) / f"{img_name}.jpg").resolve()


# --- 1) Per-scenario swap needs (from SC slot math) ---
swap_needs_rows = []
for pct in SC_SHARE_PCTS:
    counts = sc_slot_counts(pct)
    swap_needs_rows.append(
        {
            "scenario": f"sc_{pct}pct",
            "sc_share_pct": pct,
            "sc_waste_swaps": counts["sc_waste_swap"],
            "sc_background_swaps": counts["sc_bg_swap"],
            "gsvi_total_swaps": counts["sc_total_swap"],
        }
    )

swap_needs_df = pd.DataFrame(swap_needs_rows)
max_waste_swaps = int(swap_needs_df["sc_waste_swaps"].max())
max_bg_swaps = int(swap_needs_df["sc_background_swaps"].max())
max_total_swaps = int(swap_needs_df["gsvi_total_swaps"].max())

display(Markdown("### GSVI replacement counts per scenario"))
display(
    swap_needs_df.style.format(
        {
            "sc_share_pct": "{:.0f}",
            "sc_waste_swaps": "{:,.0f}",
            "sc_background_swaps": "{:,.0f}",
            "gsvi_total_swaps": "{:,.0f}",
        }
    ).hide(axis="index")
)

print(
    f"\nMax GSVI replacements in any scenario: {max_total_swaps:,} "
    f"({max_waste_swaps:,} waste + {max_bg_swaps:,} background) — at sc_0pct"
)

# --- 2) Required GSVI replacements — pick directly from pools (no scenarios) ---
required_w = pick_from_pool(
    replacement_pools["gsvi_waste"], max_waste_swaps, RANDOM_SEED, "gsvi_waste_required"
)
required_w["label_role"] = "required"

required_b = pick_from_pool(
    replacement_pools["gsvi_background"], max_bg_swaps, RANDOM_SEED, "gsvi_background_required"
)
required_b["label_role"] = "required"

required_replacements = (
    pd.concat([required_w, required_b], ignore_index=True)
    .sort_values(["class", "img_name"])
    .reset_index(drop=True)
)

required_waste = (required_replacements["class"] == "waste").sum()
required_bg = (required_replacements["class"] == "background").sum()

print(
    f"Required GSVI replacements (pool picks): {len(required_replacements):,} "
    f"({required_waste:,} waste + {required_bg:,} background)"
)
assert required_waste == max_waste_swaps
assert required_bg == max_bg_swaps

# --- 3) Buffer extras on top (for misclassification) ---
label_waste_n = buffered_count(max_waste_swaps, LABEL_BUFFER_PCT)
label_bg_n = buffered_count(max_bg_swaps, LABEL_BUFFER_PCT)
buffer_waste_n = label_waste_n - max_waste_swaps
buffer_bg_n = label_bg_n - max_bg_swaps

required_names = set(required_replacements["img_name"])
buffer_parts = []

if buffer_waste_n:
    waste_pool = replacement_pools["gsvi_waste"][
        ~replacement_pools["gsvi_waste"]["img_name"].isin(required_names)
    ]
    buffer_w = pick_from_pool(waste_pool, buffer_waste_n, RANDOM_SEED + 1, "gsvi_waste_buffer")
    buffer_w["label_role"] = "buffer"
    buffer_parts.append(buffer_w)

if buffer_bg_n:
    bg_pool = replacement_pools["gsvi_background"][
        ~replacement_pools["gsvi_background"]["img_name"].isin(required_names)
    ]
    buffer_b = pick_from_pool(bg_pool, buffer_bg_n, RANDOM_SEED + 1, "gsvi_background_buffer")
    buffer_b["label_role"] = "buffer"
    buffer_parts.append(buffer_b)

buffer_replacements = (
    pd.concat(buffer_parts, ignore_index=True) if buffer_parts else required_replacements.iloc[0:0].copy()
)

label_set = pd.concat([required_replacements, buffer_replacements], ignore_index=True)
label_waste = (label_set["class"] == "waste").sum()
label_bg = (label_set["class"] == "background").sum()

display(Markdown(f"### Label set with {LABEL_BUFFER_PCT:.0%} buffer"))
display(
    pd.DataFrame(
        [
            {
                "class": "waste",
                "required": max_waste_swaps,
                "buffer_extra": buffer_waste_n,
                "total_to_label": label_waste_n,
            },
            {
                "class": "background",
                "required": max_bg_swaps,
                "buffer_extra": buffer_bg_n,
                "total_to_label": label_bg_n,
            },
            {
                "class": "total",
                "required": max_total_swaps,
                "buffer_extra": buffer_waste_n + buffer_bg_n,
                "total_to_label": label_waste_n + label_bg_n,
            },
        ]
    ).style.format({"required": "{:,.0f}", "buffer_extra": "{:,.0f}", "total_to_label": "{:,.0f}"}).hide(axis="index")
)

# --- 4) Resolve archive paths + build copy manifest ---
label_rows = []
for _, row in label_set.iterrows():
    src = gsvi_archive_path(row["img_name"], row["img_dir"])
    dest_dir = SENSITIVITY_WASTE_DIR if row["class"] == "waste" else SENSITIVITY_BG_DIR
    dest = dest_dir / f"{row['img_name']}.jpg"
    label_rows.append(
        {
            "img_name": row["img_name"],
            "class": row["class"],
            "label_role": row["label_role"],
            "collection": row["collection"],
            "img_dir": row["img_dir"],
            "src_path": str(src),
            "dest_path": str(dest),
            "src_exists": src.exists(),
        }
    )

label_manifest_df = pd.DataFrame(label_rows)
missing_src = (~label_manifest_df["src_exists"]).sum()

print(f"\nSource images found on disk: {label_manifest_df['src_exists'].sum():,} / {len(label_manifest_df):,}")
if missing_src:
    print(f"WARNING: {missing_src:,} source images missing — check paths before copying")

swap_needs_df.to_csv(SWAP_NEEDS_CSV, index=False)
label_manifest_df.to_csv(LABEL_MANIFEST_CSV, index=False)

print(f"\nSaved swap needs: {SWAP_NEEDS_CSV}")
print(f"Saved label manifest: {LABEL_MANIFEST_CSV}")
print(f"\nCopy destinations:")
print(f"  waste:       {SENSITIVITY_WASTE_DIR} ({label_waste:,} images: {required_waste:,} required + {buffer_waste_n:,} buffer)")
print(f"  background:  {SENSITIVITY_BG_DIR} ({label_bg:,} images: {required_bg:,} required + {buffer_bg_n:,} buffer)")

### GSVI replacement counts per scenario

scenario,sc_share_pct,sc_waste_swaps,sc_background_swaps,gsvi_total_swaps
sc_0pct,0,38,47,85
sc_25pct,25,28,35,63
sc_50pct,50,19,23,42
sc_75pct,75,10,12,22
sc_100pct,100,0,0,0



Max GSVI replacements in any scenario: 85 (38 waste + 47 background) — at sc_0pct
Required GSVI replacements (pool picks): 85 (38 waste + 47 background)


### Label set with 40% buffer

class,required,buffer_extra,total_to_label
waste,38,16,54
background,47,19,66
total,85,35,120



Source images found on disk: 120 / 120

Saved swap needs: /Users/wenlanzhang/PycharmProjects/Chapter_waste/6_sensitivity/Train0315_695_gsvi_swap_needs.csv
Saved label manifest: /Users/wenlanzhang/PycharmProjects/Chapter_waste/6_sensitivity/Train0315_695_gsvi_label_manifest.csv

Copy destinations:
  waste:       /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/waste (54 images: 38 required + 16 buffer)
  background:  /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/background (66 images: 47 required + 19 buffer)


In [ ]:
import shutil

COPY_IMAGES = False  # set True when ready to copy from Google archive

if COPY_IMAGES:
    SENSITIVITY_WASTE_DIR.mkdir(parents=True, exist_ok=True)
    SENSITIVITY_BG_DIR.mkdir(parents=True, exist_ok=True)

    copied, skipped = 0, 0
    for _, row in label_manifest_df.iterrows():
        src = Path(row["src_path"])
        dest = Path(row["dest_path"])
        if not src.exists():
            skipped += 1
            continue
        if dest.exists():
            skipped += 1
            continue
        shutil.copy2(src, dest)
        copied += 1

    print(f"Copied {copied:,} images | skipped {skipped:,} (missing or already present)")
    print(f"  waste:      {SENSITIVITY_WASTE_DIR}")
    print(f"  background: {SENSITIVITY_BG_DIR}")
else:
    print("COPY_IMAGES = False — run with COPY_IMAGES = True to copy images to sensitivity folders")

Copied 8 images | skipped 112 (missing or already present)
  waste:      /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/waste
  background: /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/background


## 9. Build scenario training folders (post-labelling)

After labelling GSVI replacements, build each sensitivity scenario as a **YOLO-ready folder** under `sensitivity/sc_*pct/`.

Each folder contains:
- `images/` + `labels/` — 695 images (610 GSVI core + kept/swapped SC slots)
- `train.txt` — absolute paths for YOLO training
- `Waste.yaml` — dataset config
- `training.csv` — 695-row manifest for this scenario

**Replacement pools (labelled):**
- **Waste swaps** → `20260708_label_waste` images with non-empty labels (confirmed waste)
- **Background swaps** → `background/` batch + any re-labelled-as-background images (empty label)

Run section 7 first (or ensure `Train0315_695_training.csv` exists).

In [19]:
import shutil
from IPython.display import Markdown, display

SENSITIVITY_IMG_ROOT = IMG_ROOT / "sensitivity"
LABELED_WASTE_DIR = SENSITIVITY_IMG_ROOT / "20260708_label_waste"
LABELED_BG_DIR = SENSITIVITY_IMG_ROOT / "background"
SCENARIO_ROOT = SENSITIVITY_IMG_ROOT
SCENARIO_CSV_DIR = OUTPUT_DIR / "scenarios"
BUILD_SCENARIOS = True


def canonical_img_name(stem: str) -> str:
    return stem.split("-", 1)[1] if UUID_PREFIX.match(stem) else stem


def resolve_train_files(img_name: str) -> tuple[Path, Path, str]:
    for cand in candidate_names(img_name):
        img = IMAGES_DIR / f"{cand}.jpg"
        lbl = LABELS_DIR / f"{cand}.txt"
        if img.exists() and lbl.exists():
            return img, lbl, cand
    raise FileNotFoundError(f"Missing Train0315_695 files for {img_name}")


def load_labeled_replacement_pools() -> dict[str, pd.DataFrame]:
    waste_rows, bg_rows = [], []

    for img_path in sorted((LABELED_WASTE_DIR / "images").glob("*.jpg")):
        file_stem = img_path.stem
        label_path = LABELED_WASTE_DIR / "labels" / f"{file_stem}.txt"
        has_waste = label_path.exists() and label_path.stat().st_size > 0
        row = {
            "img_name": canonical_img_name(file_stem),
            "file_stem": file_stem,
            "class": "waste" if has_waste else "background",
            "collection": "GSVI",
            "image_path": str(img_path),
            "label_path": str(label_path) if has_waste else None,
            "has_local": True,
            "candidate_source": "labeled",
        }
        (waste_rows if has_waste else bg_rows).append(row)

    for img_path in sorted(LABELED_BG_DIR.glob("*.jpg")):
        file_stem = img_path.stem
        bg_rows.append(
            {
                "img_name": canonical_img_name(file_stem),
                "file_stem": file_stem,
                "class": "background",
                "collection": "GSVI",
                "image_path": str(img_path),
                "label_path": None,
                "has_local": True,
                "candidate_source": "background_batch",
            }
        )

    return {
        "gsvi_waste": pd.DataFrame(waste_rows).drop_duplicates("img_name"),
        "gsvi_background": pd.DataFrame(bg_rows).drop_duplicates("img_name"),
    }


def build_labeled_lookup(pools: dict[str, pd.DataFrame]) -> dict[str, dict]:
    lookup: dict[str, dict] = {}
    for pool_df in pools.values():
        for _, row in pool_df.iterrows():
            lookup[row["img_name"]] = row.to_dict()
    return lookup


def build_swap_scenario_labeled(
    sc_share_pct: int,
    labeled_pools: dict[str, pd.DataFrame],
    seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    """Keep all 610 GSVI core; swap SC slots using post-labelling GSVI pools."""
    counts = sc_slot_counts(sc_share_pct)
    parts = []

    core = gsvi_core.copy()
    core["swap_role"] = "gsvi_core"
    parts.append(core)

    sc_waste_kept = pick_training_rows(sc_waste_training, counts["sc_waste_keep"], seed)
    sc_waste_kept["swap_role"] = "sc_kept"
    parts.append(sc_waste_kept)

    sc_bg_kept = pick_training_rows(sc_bg_training, counts["sc_bg_keep"], seed)
    sc_bg_kept["swap_role"] = "sc_kept"
    parts.append(sc_bg_kept)

    if counts["sc_waste_swap"]:
        repl_w = pick_from_pool(
            labeled_pools["gsvi_waste"], counts["sc_waste_swap"], seed, "labeled_gsvi_waste"
        )
        repl_w["swap_role"] = "gsvi_replacement"
        parts.append(repl_w)

    if counts["sc_bg_swap"]:
        repl_b = pick_from_pool(
            labeled_pools["gsvi_background"], counts["sc_bg_swap"], seed, "labeled_gsvi_background"
        )
        repl_b["swap_role"] = "gsvi_replacement"
        parts.append(repl_b)

    scenario = pd.concat(parts, ignore_index=True)
    scenario["sc_share_pct"] = sc_share_pct
    scenario["scenario"] = f"sc_{sc_share_pct}pct"

    assert len(scenario) == TOTAL_IMAGES
    assert (scenario["class"] == "waste").sum() == WASTE_TOTAL
    assert (scenario["class"] == "background").sum() == BG_TOTAL
    return scenario


def materialize_scenario_folder(
    scenario_df: pd.DataFrame,
    scenario_dir: Path,
    labeled_lookup: dict[str, dict],
) -> int:
    images_dir = scenario_dir / "images"
    labels_dir = scenario_dir / "labels"

    if scenario_dir.exists():
        shutil.rmtree(scenario_dir)
    images_dir.mkdir(parents=True)
    labels_dir.mkdir(parents=True)

    train_paths: list[str] = []
    for _, row in scenario_df.iterrows():
        if row["swap_role"] in ("gsvi_core", "sc_kept"):
            src_img, src_lbl, out_stem = resolve_train_files(row["img_name"])
        else:
            info = labeled_lookup[row["img_name"]]
            src_img = Path(info["image_path"])
            label_path = info.get("label_path")
            src_lbl = Path(label_path) if label_path else None
            out_stem = info["file_stem"]

        dest_img = images_dir / f"{out_stem}.jpg"
        dest_lbl = labels_dir / f"{out_stem}.txt"
        shutil.copy2(src_img, dest_img)

        if src_lbl and src_lbl.exists():
            shutil.copy2(src_lbl, dest_lbl)
        else:
            dest_lbl.write_text("")

        train_paths.append(str(dest_img.resolve()))

    train_txt = scenario_dir / "train.txt"
    train_txt.write_text("\n".join(train_paths) + "\n")

    waste_yaml = scenario_dir / "Waste.yaml"
    waste_yaml.write_text(
        f"train: {train_txt}\n"
        f"val: {train_txt}\n\n"
        f"nc: 1\n\n"
        f"names: ['Waste']\n"
    )
    return len(train_paths)


# --- Load labelled replacement pools ---
labeled_pools = load_labeled_replacement_pools()
labeled_lookup = build_labeled_lookup(labeled_pools)

print("Labelled replacement pools:")
print(
    f"  waste: {len(labeled_pools['gsvi_waste']):,} "
    f"(need ≤ {sc_slot_counts(0)['sc_waste_swap']:,} per scenario)"
)
print(
    f"  background: {len(labeled_pools['gsvi_background']):,} "
    f"(need ≤ {sc_slot_counts(0)['sc_bg_swap']:,} per scenario)"
)

# --- Build scenarios ---
scenario_tables: dict[str, pd.DataFrame] = {}
scenario_rows = []
baseline_names = set(train_cmp["img_name"])

for pct in SC_SHARE_PCTS:
    tag = f"sc_{pct}pct"
    scenario_df = build_swap_scenario_labeled(pct, labeled_pools)
    scenario_tables[tag] = scenario_df

    scenario_names = set(scenario_df["img_name"])
    counts = sc_slot_counts(pct)

    scenario_rows.append(
        {
            "scenario": tag,
            "sc_share_pct": pct,
            "sc_kept": counts["sc_total_keep"],
            "sc_swapped_to_gsvi": counts["sc_total_swap"],
            "gsvi_images": (scenario_df["collection"] == "GSVI").sum(),
            "sc_images": (scenario_df["collection"] == "SC").sum(),
            "gsvi_replacements": (scenario_df["swap_role"] == "gsvi_replacement").sum(),
            "total_images": len(scenario_df),
            "swapped_out": len(baseline_names - scenario_names),
            "swapped_in": len(scenario_names - baseline_names),
        }
    )

scenario_summary = pd.DataFrame(scenario_rows)

display(Markdown("### Scenario summary (post-labelling)"))
display(
    scenario_summary.style.format(
        {
            "sc_share_pct": "{:.0f}",
            "sc_kept": "{:,.0f}",
            "sc_swapped_to_gsvi": "{:,.0f}",
            "gsvi_images": "{:,.0f}",
            "sc_images": "{:,.0f}",
            "gsvi_replacements": "{:,.0f}",
            "total_images": "{:,.0f}",
            "swapped_out": "{:,.0f}",
            "swapped_in": "{:,.0f}",
        }
    ).hide(axis="index")
)

# --- Write folders + CSVs ---
if BUILD_SCENARIOS:
    SCENARIO_CSV_DIR.mkdir(parents=True, exist_ok=True)

    for tag, scenario_df in scenario_tables.items():
        scenario_dir = SCENARIO_ROOT / tag
        n_images = materialize_scenario_folder(scenario_df, scenario_dir, labeled_lookup)

        scenario_df.to_csv(SCENARIO_CSV_DIR / f"{tag}_training.csv", index=False)
        scenario_df.to_csv(scenario_dir / "training.csv", index=False)

        print(
            f"{tag}: {n_images:,} images → {scenario_dir} | "
            f"GSVI {(scenario_df['collection']=='GSVI').sum():,} / "
            f"SC {(scenario_df['collection']=='SC').sum():,}"
        )

    scenario_summary.to_csv(SCENARIO_CSV_DIR / "scenario_summary.csv", index=False)
    print(f"\nScenario folders: {SCENARIO_ROOT}/sc_*pct/")
    print(f"Scenario CSVs: {SCENARIO_CSV_DIR}/")
else:
    print("BUILD_SCENARIOS = False — set True to write training folders")

Labelled replacement pools:
  waste: 39 (need ≤ 38 per scenario)
  background: 62 (need ≤ 47 per scenario)


### Scenario summary (post-labelling)

scenario,sc_share_pct,sc_kept,sc_swapped_to_gsvi,gsvi_images,sc_images,gsvi_replacements,total_images,swapped_out,swapped_in
sc_0pct,0,0,85,695,0,85,695,85,85
sc_25pct,25,22,63,673,22,63,695,63,63
sc_50pct,50,43,42,652,43,42,695,42,42
sc_75pct,75,63,22,632,63,22,695,22,22
sc_100pct,100,85,0,610,85,0,695,0,0


sc_0pct: 695 images → /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/sc_0pct | GSVI 695 / SC 0
sc_25pct: 695 images → /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/sc_25pct | GSVI 673 / SC 22
sc_50pct: 695 images → /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/sc_50pct | GSVI 652 / SC 43
sc_75pct: 695 images → /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/sc_75pct | GSVI 632 / SC 63
sc_100pct: 695 images → /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/sc_100pct | GSVI 610 / SC 85

Scenario folders: /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/sensitivity/sc_*pct/
Scenario CSVs: /Users/wenlanzhang/PycharmProjects/Chapter_waste/6_sensitivity/scenarios/
